# Coding Sequence Extraction and Visualization

Extract coding DNA sequences (CDS) from genbank files and export to .fasta files; this is the format 10X wants.

In [1]:
from pathlib import Path
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

In [2]:
base_dir = Path('/home/workspace/drg/misc/sequences')

aav_dir = base_dir / 'virus'
mouse_dir = base_dir / 'mouse'

output_dir = base_dir / 'fasta'
output_dir.mkdir(parents = True, exist_ok = True)

In [3]:
extractions = {
    aav_dir / "egfp_addgene-plasmid-50465-sequence-240907.gbk": [
        ("aav_egfp", 653, 1372)
    ],
    aav_dir / "cre_addgene-plasmid-105553-sequence-202451.gbk": [
        ("aav_cre", 597, 1628)
    ],
    aav_dir / "mcherry_addgene-plasmid-114472-sequence-220931.gbk": [
        ("aav_mcherry", 642, 1349)
    ],
    aav_dir / "flp0_addgene-plasmid-51669-sequence-305863.gbk": [
        ("aav_flp0", 649, 1947)
    ],
    mouse_dir / "ai224_mouse_addgene-plasmid-181760-sequence-356681.gbk": [
        ("ai224_egfp", 7291, 8007),
        ("ai224_tdtomato", 13430, 14128)
    ],
}

In [4]:
output_dir = Path("/home/workspace/drg/misc/sequences/fasta")
output_dir.mkdir(parents=True, exist_ok=True)

for gb_path_str, targets in extractions.items():
    gb_path = Path(gb_path_str)
    record = SeqIO.read(gb_path, "genbank")

    for tag, start, end in targets:
        seq = record.seq[start - 1:end]  # 1-based inclusive
        fasta_record = SeqRecord(seq.upper(), id=tag, description=f"{tag} extracted from {gb_path.name}")
        output_file = output_dir / f"{tag}.fasta"
        SeqIO.write(fasta_record, output_file, "fasta")
        print(f"Saved to: {output_file}")

Saved to: /home/workspace/drg/misc/sequences/fasta/aav_egfp.fasta
Saved to: /home/workspace/drg/misc/sequences/fasta/aav_cre.fasta
Saved to: /home/workspace/drg/misc/sequences/fasta/aav_mcherry.fasta
Saved to: /home/workspace/drg/misc/sequences/fasta/aav_flp0.fasta
Saved to: /home/workspace/drg/misc/sequences/fasta/ai224_egfp.fasta
Saved to: /home/workspace/drg/misc/sequences/fasta/ai224_tdtomato.fasta


# Confirm exact matches

Function to verify this. Copy and paste bold print output into `.gbk` files from the websites in VSCode to confirm accuracy of exact matches. Use the extraction dictionary above. I've confirmed they are exact matches but will likely continue to confirm each time manually

In [16]:
def format_genbank_with_bold(seq, start, highlight_start, highlight_end):
    """
    Format a sequence in GenBank style (60 chars/line, 10-char groups), 
    using ANSI bold+underline for the region between highlight_start and highlight_end (1-based).
    """
    lines = []
    seq = seq.lower()
    for i in range(0, len(seq), 60):
        line_start = start + i
        chunk = seq[i:i+60]
        grouped = []
        for j in range(0, len(chunk), 10):
            word = chunk[j:j+10]
            word_start = line_start + j
            word_end = word_start + len(word) - 1

            if word_end < highlight_start or word_start > highlight_end:
                grouped.append(word)
            else:
                highlighted = ''
                for k, base in enumerate(word):
                    pos = word_start + k
                    if highlight_start <= pos <= highlight_end:
                        highlighted += f'\033[1;4m{base}\033[0m'  # Bold + Underline
                    else:
                        highlighted += base
                grouped.append(highlighted)

        lines.append(f"{line_start:>9} {' '.join(grouped)}")
    return '\n'.join(lines)

In [17]:
# Processing all extractions
for gbk_path, regions in extractions.items():
    record = SeqIO.read(gbk_path, "genbank")
    full_seq = str(record.seq)

    for label, h_start, h_end in regions:
        print(f"\n>{label} from {gbk_path} ({h_start}..{h_end})")
        formatted = format_genbank_with_bold(full_seq, 1, h_start, h_end)
        print(formatted)


>aav_egfp from /home/workspace/drg/misc/sequences/virus/egfp_addgene-plasmid-50465-sequence-240907.gbk (653..1372)
        1 catgtcctgc aggcagctgc gcgctcgctc gctcactgag gccgcccggg cgtcgggcga
       61 cctttggtcg cccggcctca gtgagcgagc gagcgcgcag agagggagtg gccaactcca
      121 tcactagggg ttcctgcggc cgcacgcgtg tgtctagact gcagagggcc ctgcgtatga
      181 gtgcaagtgg gttttaggac caggatgagg cggggtgggg gtgcctacct gacgaccgac
      241 cccgacccac tggacaagca cccaaccccc attccccaaa ttgcgcatcc cctatcagag
      301 agggggaggg gaaacaggat gcggcgaggc gcgtgcgcac tgccagcttc agcaccgcgg
      361 acagtgcctt cgcccccgcc tggcggcgcg cgccaccgcc gcctcagcac tgaaggcgcg
      421 ctgacgtcac tcgccggtcc cccgcaaact ccccttcccg gccaccttgg tcgcgtccgc
      481 gccgccgccg gcccagccgg accgcaccac gcgaggcgcg agataggggg gcacgggcgc
      541 gaccatctgc gctgcggcgc cggcgactca gcgctgcctc agtctgcggt gggcagcgga
      601 ggagtcgtgt cgtgcctgag agcgcagtcg agaaggtacc ggatccgcca ccatggtgag
      661 caagggcgag gagctgttca ccggggtggt gccca